# Interactive COBOL Conversion

This notebook shows the Python workflow for migration engineers: load a COBOL source file, expand COPY members, inspect diagnostics, and preview generated Rust.

In [ ]:
from pathlib import Path
from pprint import pprint

import cobol_converter

source_path = Path("../samples/payroll/cobol/payroll.cbl")
copybook_root = Path("../samples/payroll/copybooks")

source = source_path.read_text(encoding="utf-8")
copybooks = {
    path.name: path.read_text(encoding="utf-8")
    for path in copybook_root.glob("*.cpy")
}

print(f"Loaded {source_path} with {len(copybooks)} copybooks")

In [ ]:
expanded = cobol_converter.preprocess(
    source,
    copybooks,
    source_format="free",
)

print(expanded[:1200])

In [ ]:
check = cobol_converter.check_cobol(
    source,
    "ibm_zos",
    {"source_format": "free", "copybooks": copybooks},
)

diagnostics = check.get("diagnostics", [])
print("ok:", check.get("ok"))
pprint(diagnostics[:10])

In [ ]:
result = cobol_converter.convert_cobol(
    source,
    "ibm_zos",
    {"source_format": "free", "copybooks": copybooks},
)

if result.get("ok"):
    print(result["rust"][:2000])
else:
    print("Conversion blocked by diagnostics")
    pprint(result.get("diagnostics", [])[:10])